# XAI and Cancer Screening

## Load Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
import os
import joblib
import shap 

fp_data_folder = "../data"
fp_checkpoint_folder = "../tree_featsel_checkpoints"
fp_model_checkpoint_folder = os.path.join(fp_checkpoint_folder, "models")
fp_preprocessing_checkpoint_folder = os.path.join(fp_checkpoint_folder, "preprocessing")
fp_tuning_checkpoint_folder = os.path.join(fp_checkpoint_folder, "tuning")
fp_results_checkpoint_folder = os.path.join(fp_checkpoint_folder, "results")
fp_lung_data_file = os.path.join(fp_data_folder, "schs_200lung.csv")
fp_crc_data_file = os.path.join(fp_data_folder, "colorectalcancers_schs_pgs.csv")

fp_crc_scaler_file = os.path.join(fp_preprocessing_checkpoint_folder, "crc_minmax_scaler.pickle")
fp_crc_encoder_file = os.path.join(fp_preprocessing_checkpoint_folder, "crc_encoder.pickle")
fp_crc_tuning_w_pc_file = os.path.join(fp_tuning_checkpoint_folder, "crc_tuning_w_pc.csv")
fp_crc_tuning_wo_pc_file = os.path.join(fp_tuning_checkpoint_folder, "crc_tuning_wo_pc.csv")

seed = 2024

## Load Data

In [ ]:
crc_df = pd.read_csv(fp_crc_data_file, index_col=0)
crc_df

In [ ]:
crc_df.columns

In [ ]:
crc_df['aMED'].describe() # Assume this is also a index (continuous)

In [ ]:
crc_df['DASH'].describe() # Assume this is a index (continuous)

### Data Exploration

In [ ]:
def get_col_types_crc(df):
    categorical_cols = [
        "Sex (1=Male, 2=Female)", "alcohol_DailyandWeekly(1)vsMonthlyandNonDrinkers(0)", 
        "smoke_never(0)_ex(1)_current(2)", "Prevalent_diabetes"]
    target_col = "colorectal cancer"
    continuous_cols = [col for col in df.columns if col not in categorical_cols + [target_col]]
    print("Categorical Columns:", categorical_cols)
    print("Continuous Columns:", continuous_cols)
    print("Target Column:", target_col)
    return categorical_cols, continuous_cols, target_col

crc_categorical_cols, crc_continuous_cols, crc_target_col = get_col_types_crc(crc_df)

In [ ]:
crc_df[crc_continuous_cols].describe()

In [ ]:
def show_continuous_features_crc(df, continuous_cols, clean_col_names):
    from math import ceil
    nrows = 2
    ncols = ceil(len(continuous_cols)/2)
    size = 2
    fig, axes = plt.subplots(nrows, ncols, dpi=300, figsize=(size*ncols, size*nrows))
    axes = axes.flatten()
    axes[0].set_ylabel("Count")
    for i, col in enumerate(continuous_cols):
        df[col].hist(bins=10, ax=axes[i])
        axes[i].set_xlabel(clean_col_names[i])
    for j in range(i+1, ncols*nrows):
        axes[j].set_axis_off()
    plt.tight_layout()
    
show_continuous_features_crc(
    crc_df, crc_continuous_cols[1:], 
    ["Age", "PC1", "PC2", "PC3", "BMI", "Telomere Length", "aHEI2010\nDietary Score", 
     'aMED', 'DASH', "SBP", "DBP", "Leisure\nScreen Time", 'z_pgs000055', 'z_pgs000734']
)      

In [ ]:
for col in crc_categorical_cols + [crc_target_col]:
    print(crc_df[col].value_counts())

### Data Processing

#### Drop NA

In [ ]:
def drop_na(df):
    df = df.copy()
    display(df.isna().sum())
    print(f"DataFrame Shape Before Dropping NA: {df.shape}")
    df = df.dropna()
    print(f"DataFrame Shape Before Dropping NA: {df.shape}")
    return df
crc_df = drop_na(crc_df)
display(crc_df)

In [ ]:
for col in crc_categorical_cols + [crc_target_col]:
    print(crc_df[col].value_counts())

#### Data Split

In [ ]:
# Split data
def split_data_stratified(df, target_col, val_prop, test_prop, seed):
    np.random.seed(seed)
    df = df.copy()
    data_size = len(df)
    train_prop = 1 - val_prop - test_prop
    train_dfs, val_dfs, test_dfs = [], [], []
    # From each class, choose the same proportion of val and test
    for target, group_df in df.groupby(target_col):
        group_size = len(group_df)
        indices = np.array([i for i in range(group_size)])
        np.random.shuffle(indices)
        train_size, val_size = round(train_prop * group_size), round(val_prop * group_size)
        train_indices, val_indices, test_indices = (
            indices[:train_size], indices[train_size:train_size+val_size], indices[train_size+val_size:])
        train_dfs.append(group_df.iloc[train_indices])
        val_dfs.append(group_df.iloc[val_indices])
        test_dfs.append(group_df.iloc[test_indices])
    return dict(
        train_df=pd.concat(train_dfs), val_df=pd.concat(val_dfs), test_df=pd.concat(test_dfs)
    )

crc_split_dict = split_data_stratified(crc_df, crc_target_col, val_prop=0.1, test_prop=0.1, seed=2023)
for label, cur_df in crc_split_dict.items():
    print(label,":")
    print("- Total Number of Samples:", len(cur_df))
    print("- Class Distribution:", cur_df[crc_target_col].value_counts()/len(cur_df))

#### Data Normalisation

In [ ]:
def normalise_data(train_df, val_df, test_df, continuous_cols):
    train_df, val_df, test_df = train_df.copy(), val_df.copy(), test_df.copy()
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()
    train_df[continuous_cols] = scaler.fit_transform(train_df[continuous_cols])
    val_df[continuous_cols] = scaler.transform(val_df[continuous_cols])
    test_df[continuous_cols] = scaler.transform(test_df[continuous_cols])
    return dict(
        train_df=train_df, val_df=val_df, test_df=test_df
    ), scaler

crc_split_dict_scaled, crc_scaler = normalise_data(**crc_split_dict, continuous_cols=crc_continuous_cols)
for label, cur_df in crc_split_dict_scaled.items():
    print(label,":")
    display(cur_df[crc_continuous_cols].describe())

In [ ]:
# Save scaler
joblib.dump(crc_scaler, fp_crc_scaler_file) 

#### Binarisation + One-Hot Encoding

In [ ]:
def one_hot_encode_data(train_df, val_df, test_df, categorical_cols):
    train_df, val_df, test_df = train_df.copy(), val_df.copy(), test_df.copy()
    
    # Binarise "Sex" col
    prev_col_name = "Sex (1=Male, 2=Female)"
    new_col_name = "Sex (0=Male, 1=Female)"
    train_df[new_col_name] = train_df[prev_col_name] - 1
    val_df[new_col_name] = val_df[prev_col_name] - 1
    test_df[new_col_name] = test_df[prev_col_name] - 1

    # One-Hot Encode "smoke_never" col
    prev_col_name =  'smoke_never(0)_ex(1)_current(2)'
    new_col_names = ["smoke_ex(1)", "smoke_current(2)"]
    from sklearn.preprocessing import OneHotEncoder
    encoder = OneHotEncoder(drop="first")
    train_df[new_col_names] = encoder.fit_transform(train_df[[prev_col_name]]).toarray()
    val_df[new_col_names] = encoder.transform(val_df[[prev_col_name]]).toarray()
    test_df[new_col_names] = encoder.transform(test_df[[prev_col_name]]).toarray()
    
    train_df = train_df.drop(["Sex (1=Male, 2=Female)", 'smoke_never(0)_ex(1)_current(2)'], axis=1)
    val_df = val_df.drop(["Sex (1=Male, 2=Female)", 'smoke_never(0)_ex(1)_current(2)'], axis=1)
    test_df = test_df.drop(["Sex (1=Male, 2=Female)", 'smoke_never(0)_ex(1)_current(2)'], axis=1)
    
    return dict(
        train_df=train_df, val_df=val_df, test_df=test_df
    ), encoder

crc_split_dict_encoded, crc_encoder = one_hot_encode_data(
    **crc_split_dict_scaled, categorical_cols=crc_categorical_cols)

for label, cur_df in crc_split_dict_encoded.items():
    print(label,":")
    display(cur_df)

In [ ]:
# Save encoder
joblib.dump(crc_encoder, fp_crc_encoder_file) 

In [ ]:
# Check that continuous variables have been scaled
for label, cur_df in crc_split_dict_encoded.items():
    print(label,":")
    display(cur_df[crc_continuous_cols].describe())

In [ ]:
# Check that categorical variables correspond to pre-processed distributions
def get_new_col_types_crc(df):
    categorical_cols = [
        "Sex (0=Male, 1=Female)", "alcohol_DailyandWeekly(1)vsMonthlyandNonDrinkers(0)", "smoke_ex(1)", "smoke_current(2)", "Prevalent_diabetes"]
    target_col = "colorectal cancer"
    continuous_cols = [col for col in df.columns if col not in categorical_cols + [target_col]]
    print("Categorical Columns:", categorical_cols)
    print("Continuous Columns:", continuous_cols)
    print("Target Column:", target_col)
    return categorical_cols, continuous_cols, target_col

crc_categorical_cols_new, crc_continuous_cols_new, crc_target_col_new = get_new_col_types_crc(
    crc_split_dict_encoded["train_df"])

In [ ]:
testing_df = pd.concat(list(crc_split_dict_encoded.values()))
for col in crc_categorical_cols_new + [crc_target_col_new]:
    print(testing_df[col].value_counts())

#### Oversample Data

In [ ]:
def oversample_data(train_df, val_df, test_df, target_col):
    from imblearn.over_sampling import SMOTENC
    train_df, val_df, test_df = train_df.copy(), val_df.copy(), test_df.copy()
    oversample = SMOTENC(categorical_features=crc_categorical_cols_new, random_state=seed)
    feature_cols = [col for col in train_df.columns if col != target_col]
    X, y = oversample.fit_resample(train_df[feature_cols], train_df[target_col])
    train_df = pd.DataFrame(X, columns=feature_cols)
    train_df[target_col] = y
    return dict(
        train_df=train_df, val_df=val_df, test_df=test_df
    )

crc_split_dict_oversampled = oversample_data(
    **crc_split_dict_encoded, target_col=crc_target_col_new)
crc_split_dict_oversampled["train_df"][crc_target_col_new].value_counts()

In [ ]:
crc_feat_cols_w_pc = crc_continuous_cols_new + crc_categorical_cols_new
crc_feat_cols_w_pc = [e for e in crc_feat_cols_w_pc if (e!="Follow-up time")]
print(f"{len(crc_feat_cols_w_pc)} Feature Columns With PCs: {crc_feat_cols_w_pc}")
crc_feat_cols_wo_pc = crc_continuous_cols_new + crc_categorical_cols_new
crc_feat_cols_wo_pc = [e for e in crc_feat_cols_wo_pc if ("PC" not in e) & (e!="Follow-up time")]
print(f"{len(crc_feat_cols_wo_pc)} Feature Columns Without PCs: {crc_feat_cols_wo_pc}")

### Training & Tuning Functions

In [ ]:
def evaluate(Y, Y_pred_prob, label):
    from sklearn.metrics import log_loss, accuracy_score, roc_auc_score
    Y_pred = np.argmax(Y_pred_prob, axis=-1)
    return {
        f"{label} Accuracy": accuracy_score(Y, Y_pred),
        f"{label} AUC": roc_auc_score(Y, Y_pred_prob[:, 1]),
        f"{label} Loss": log_loss(Y, Y_pred_prob[:, 1]),
    }

def tune_model_crc(
    train_df, val_df, test_df, 
    param_grid, 
    feat_cols, target_col, seed=seed, verbose=True
):
    # from sklearn.ensemble import RandomForestClassifier
    from imblearn.ensemble import BalancedRandomForestClassifier
    from sklearn.model_selection import ParameterGrid
    
    train_df, val_df = train_df.copy(), val_df.copy()
    
    # Process data
    train_X, train_Y = train_df[feat_cols].values, train_df[target_col].values
    valid_X, valid_Y = val_df[feat_cols].values, val_df[target_col].values
    test_X, test_Y = test_df[feat_cols].values, test_df[target_col].values
    
    df_list = []
    grid = list(ParameterGrid(param_grid))
    best_val_auc = None
    # Tune model
    pbar = tqdm(grid, disable=not verbose, total=len(grid))
    for param_dict in pbar:
        if best_val_auc is None:
            pbar.set_description(f"Params: {param_dict}")
        else:
            pbar.set_description(f"Params: {param_dict}, Best Val AUC: {best_val_auc:.5f}")
        # clf = RandomForestClassifier(random_state=seed, n_estimators=n_estimators)
        # https://machinelearningmastery.com/bagging-and-random-forest-for-imbalanced-classification/
        clf = BalancedRandomForestClassifier(
            random_state=seed, **param_dict,
            sampling_strategy="all", replacement=True, bootstrap=False
        )
        start = time.time()
        clf.fit(train_X, train_Y)
        end = time.time()
        train_Y_pred = clf.predict_proba(train_X)
        valid_Y_pred = clf.predict_proba(valid_X)
        test_Y_pred = clf.predict_proba(test_X)
        
        results_dict = param_dict.copy()
        results_dict.update(evaluate(train_Y, train_Y_pred, label="Training"))
        valid_perf_dict = evaluate(valid_Y, valid_Y_pred, label="Validation")
        results_dict.update(valid_perf_dict)
        results_dict.update(evaluate(test_Y, test_Y_pred, label="Testing"))
        results_dict["Train Time/s"] = end-start
        df_list.append(results_dict)
        if ((best_val_auc is None) or (valid_perf_dict["Validation AUC"] > best_val_auc)):
            best_val_auc = valid_perf_dict["Validation AUC"]
    tuning_df = pd.DataFrame(df_list)
    best_index = tuning_df["Validation AUC"].idxmax()
    tuning_df["Best Hyperparamter"] = False
    tuning_df.loc[best_index, "Best Hyperparamter"] = True
    best_param_dict = grid[best_index]
    return best_param_dict, tuning_df


def get_predictions(train_df, val_df, test_df, feat_cols, target_col, clf, verbose=True):
    train_df, val_df = train_df.copy(), val_df.copy()
    from sklearn.metrics import log_loss, accuracy_score, roc_auc_score
    train_X, train_Y = train_df[feat_cols].values, train_df[target_col].values
    valid_X, valid_Y = val_df[feat_cols].values, val_df[target_col].values
    test_X, test_Y = test_df[feat_cols].values, test_df[target_col].values
    if verbose:
        print("2. Predicting...")
    pred_col = f"{target_col}_pred"
    start = time.time()
    train_Y_pred = clf.predict_proba(train_X)
    train_df[pred_col] = train_Y_pred[:, 1]
    if verbose:
        print(f" - Prediction for Training Set Took {time.time()-start}s")
    start = time.time()
    valid_Y_pred = clf.predict_proba(valid_X)
    val_df[pred_col] = valid_Y_pred[:, 1]
    if verbose:
        print(f" - Prediction for Validation Set Took {time.time()-start}s")
    start = time.time()
    test_Y_pred = clf.predict_proba(test_X)
    test_df[pred_col] = test_Y_pred[:, 1]
    if verbose:
        print(f" - Prediction for Test Set Took {time.time()-start}s")

    # Evaluate Model
    if verbose:
        print("3. Evaluating...")
    perf_df = pd.DataFrame(dict(
        AUC = [
            roc_auc_score(train_Y, train_df[pred_col]), 
            roc_auc_score(valid_Y, val_df[pred_col]), 
            roc_auc_score(test_Y, test_df[pred_col])],
        Accuracy = [
            accuracy_score(train_Y, train_df[pred_col]>0.5), 
            accuracy_score(valid_Y, val_df[pred_col]>0.5), 
            accuracy_score(test_Y, test_df[pred_col]>0.5)],
        Loss = [
            log_loss(train_Y, train_df[pred_col]), 
            log_loss(valid_Y, val_df[pred_col]), 
            log_loss(test_Y, test_df[pred_col])], 
    ), index=["Train", "Valid", "Test"])
    
    return dict(
        train_df=train_df, val_df=val_df, test_df=test_df
    ), perf_df

def buffered_shap(explainer, matrix, chunksize, pred):
    def process_shap(shap, pred):
        pred = (pred > 0.50).values.astype(int)
        # (2, 160, 14) -> (160, 2, 14)
        shap = np.transpose(shap, (1, 0, 2))
        # Get Shap of Predicted
        shap = shap[np.arange(len(shap)), pred]
        return shap
    
    from math import ceil
    all_exp = []
    matrix_len = len(matrix)
    num_chunks = ceil(matrix_len/chunksize)
    for chunk_i in tqdm(range(num_chunks), total=num_chunks):
        start = chunk_i*chunksize
        end = (chunk_i+1)*chunksize
        if end < matrix_len:
            matrix_chunk = matrix[chunk_i*chunksize:(chunk_i+1)*chunksize]
            pred_chunk = pred[chunk_i*chunksize:(chunk_i+1)*chunksize]
        else:
            matrix_chunk = matrix[chunk_i*chunksize:]
            pred_chunk = pred[chunk_i*chunksize:]
        shap_chunk = explainer.shap_values(matrix_chunk)
        exp_chunk = process_shap(shap_chunk, pred_chunk)
        all_exp.append(exp_chunk)
    return np.vstack(all_exp)

def get_shap_values(train_df, val_df, test_df, feat_cols, target_col, clf, chunksize=64):
    train_df, val_df = train_df.copy(), val_df.copy()
    train_X, train_Y = train_df[feat_cols].values, train_df[target_col].values
    valid_X, valid_Y = val_df[feat_cols].values, val_df[target_col].values
    test_X, test_Y = test_df[feat_cols].values, test_df[target_col].values
    pred_col = f"{target_col}_pred"
    # Get Shap
    print("4. Getting Shap Values...")
    shap_cols = [f"{col}_shap" for col in feat_cols]
    print(" - Instantiating Explainer...")
    explainer = shap.TreeExplainer(clf)
    
    # (2, 160, 14)
    print("4. Getting Explanations...")
    # start = time.time()
    # train_df[shap_cols] = buffered_shap(
    #     explainer=explainer, matrix=train_X, chunksize=64, pred=train_df[pred_col]
    # )
    # print(f" - Explanations for Training Set Took {time.time()-start}s")
    # start = time.time()
    # val_df[shap_cols] = buffered_shap(
    #     explainer=explainer, matrix=valid_X, chunksize=chunksize, pred=val_df[pred_col]
    # )
    # print(f" - Explanations for Validation Set Took {time.time()-start}s")
    start = time.time()
    test_df[shap_cols]=buffered_shap(
        explainer=explainer, matrix=test_X, chunksize=64, pred=test_df[pred_col]
    )
    print(f" - Explanations for Test Set Took {time.time()-start}s")
    return dict(
        train_df=train_df, val_df=val_df, test_df=test_df
    )

def train_model_crc(
    train_df, val_df, test_df, feat_cols, target_col, param_dict, seed=seed, verbose=True):
    from imblearn.ensemble import BalancedRandomForestClassifier
    train_df, val_df, test_df = train_df.copy(), val_df.copy(), test_df.copy()
    # Process data
    train_X, train_Y = train_df[feat_cols].values, train_df[target_col].values
    valid_X, valid_Y = val_df[feat_cols].values, val_df[target_col].values
    test_X, test_Y = test_df[feat_cols].values, test_df[target_col].values
    
    # Train Model
    if verbose:
        print("1. Training The Model...")
    clf = BalancedRandomForestClassifier(
            random_state=seed, **param_dict, sampling_strategy="all", replacement=True, bootstrap=False)
    start = time.time()
    clf.fit(train_X, train_Y)
    if verbose:
        print(f" - Training Took {time.time()-start}s")
    
    return clf

def format_results(train_df, val_df, test_df, target_col, scaler, continuous_cols):
    train_df, val_df = train_df.copy(), val_df.copy()
    print("6. Generating Outputs...")
    pred_col = f"{target_col}_pred"
    pred_df = test_df.copy()
    # pred_df = pd.concat([train_df, val_df, test_df])
    pred_df[continuous_cols] = scaler.inverse_transform(pred_df[continuous_cols])
    pred_df["split"] = ["test"]*len(test_df)
    # pred_df["split"] = ["train"]*len(train_df) + ["valid"]*len(val_df)+ ["test"]*len(test_df)
    pred_bin_col = f"{target_col}_pred_bin"
    pred_df[pred_bin_col] = (pred_df[pred_col] > 0.5).astype(int)
    return pred_df

### Get Best Number of Features

In [ ]:
get_feat_sel_df(
    train_df=crc_split_dict_oversampled["train_df"], 
    feat_cols=crc_feat_cols_w_pc, 
    target_col=crc_target_col_new, 
    categorical_cols=crc_continuous_cols_new
)

In [ ]:
# Feature Selection
# https://medium.com/@hertan06/which-features-to-use-in-your-model-350630a1e31c
def get_feat_sel_df(train_df, feat_cols, target_col, categorical_cols):
    from sklearn.feature_selection import mutual_info_classif
    categorical_indices = [i for i, col in enumerate(feat_cols) if col in categorical_cols]
    # print(np.array(feat_cols)[categorical_indices])
    mi = mutual_info_classif(train_df[feat_cols].values, train_df[target_col].values, copy=True, random_state=seed)
    featsel_df = pd.DataFrame({"Feature":feat_cols, "Estimated Mutual Information":mi})
    featsel_df = featsel_df.sort_values(by="Estimated Mutual Information", ascending=False)
    return featsel_df

def feature_selection(train_df, feat_cols, target_col, categorical_cols, top_k, show_featsel_df=True):
    featsel_df = get_feat_sel_df(train_df, feat_cols, target_col, categorical_cols)
    if show_featsel_df:
        display(featsel_df)
    return featsel_df["Feature"].iloc[:top_k].values

def get_best_number_of_features(
    split_dict_encoded, split_dict_oversampled, all_feat_cols, target_col, categorical_cols
):
    df_list = []
    feat_sel_df = get_feat_sel_df(
        train_df=split_dict_oversampled["train_df"], 
        feat_cols=all_feat_cols, target_col=target_col, categorical_cols=categorical_cols)
    num_feat = len(feat_sel_df)
    all_features = feat_sel_df["Feature"].values
    
    for k in tqdm(range(1,num_feat+1), total=num_feat, position=0, leave=True):
        best_feat_oversampled = all_features[:k]
        print(f"Number of Features: {k}, Selected Features: {best_feat_oversampled}")
        best_estimators, _ = tune_model_crc(
            train_df=split_dict_encoded["train_df"], 
            val_df=split_dict_encoded["val_df"], 
            feat_cols=best_feat_oversampled, 
            target_col=target_col,
            n_estimators_list=[50, 100,200, 300, 400, 500, 600, 700], 
            seed=seed, verbose=False
        )
        print(f"   - Optimal Number of Estimators: {best_estimators}")
        # display(crc_tuning_df_w_pc)
        # crc_tuning_df_w_pc.to_csv(fp_crc_tuning_w_pc_file)
        clf = train_model_crc(
            **split_dict_encoded, feat_cols=best_feat_oversampled, target_col=target_col, 
            n_estimators=best_estimators, seed=seed, verbose=False
        )
        _, perf_df_w_pc = get_predictions(
            **split_dict_encoded, feat_cols=best_feat_oversampled, target_col=target_col,
            clf=clf, verbose=False
        )
        # display(crc_perf_df_w_pc)
        val_auc = perf_df_w_pc.loc["Valid", "AUC"]
        test_auc = perf_df_w_pc.loc["Test", "AUC"]
        df_list.append({
            "K":k, "Valid AUC":val_auc, "Test AUC":test_auc, "Num Estimators":best_estimators
        })
    df = pd.DataFrame(df_list)
    best_val_auc = df["Valid AUC"].max()
    df["Best K"] = df["Valid AUC"] == best_val_auc
    return df


In [ ]:
num_feat_df = get_best_number_of_features(
    split_dict_encoded=crc_split_dict_encoded, 
    split_dict_oversampled=crc_split_dict_oversampled, 
    all_feat_cols=crc_feat_cols_w_pc, 
    target_col=crc_target_col_new, categorical_cols=crc_continuous_cols_new
)
display(num_feat_df)

In [ ]:
num_feat_df = get_best_number_of_features(
    split_dict_encoded=crc_split_dict_encoded, 
    split_dict_oversampled=crc_split_dict_encoded, 
    all_feat_cols=crc_feat_cols_w_pc, 
    target_col=crc_target_col_new, categorical_cols=crc_continuous_cols_new
)
display(num_feat_df)

In [ ]:
best_feat_w_pc_encoded = feature_selection(
    train_df=crc_split_dict_encoded["train_df"], 
    feat_cols=crc_feat_cols_w_pc, 
    target_col=crc_target_col_new, 
    categorical_cols=crc_categorical_cols_new,
    top_k=15
)

best_feat_w_pc_oversampled = feature_selection(
    train_df=crc_split_dict_oversampled["train_df"], 
    feat_cols=crc_feat_cols_w_pc, 
    target_col=crc_target_col_new, 
    categorical_cols=crc_categorical_cols_new,
    top_k=7
)

In [ ]:
print(best_feat_w_pc_oversampled)

### Get Best Max Depth

In [ ]:
best_param_max_depth_w_pc, max_depth_tuning_df_w_pc = tune_model_crc(
    **crc_split_dict_encoded,
    feat_cols=best_feat_w_pc_oversampled, 
    target_col=crc_target_col_new,
    seed=seed,
    param_grid=dict(
        n_estimators = [50, 100,200, 300, 400, 500, 600, 700],
        # max_depth = [5, 10, 20, 30, 40]
    ), 
)

In [ ]:
max_depth_tuning_df_w_pc

In [ ]:
max_depth_tuning_df_w_pc[
    max_depth_tuning_df_w_pc.groupby("max_depth")["Validation AUC"].transform(max) == max_depth_tuning_df_w_pc["Validation AUC"]
]

In [ ]:
best_param_max_depth_w_pc = {"n_estimators":100}
crc_clf_w_pc = train_model_crc(
    **crc_split_dict_encoded, feat_cols=best_feat_w_pc_oversampled, target_col=crc_target_col_new, 
    param_dict = best_param_max_depth_w_pc, seed=seed
)
crc_pred_dict_w_pc, crc_perf_df_w_pc = get_predictions(
    **crc_split_dict_encoded, feat_cols=best_feat_w_pc_oversampled, target_col=crc_target_col_new, 
    clf=crc_clf_w_pc)
display(crc_perf_df_w_pc)

### Information for Survey

In [ ]:
class ConditionalGaussianDistribution:
    def __init__(self, Y, X):
        self.Y = Y # Assume shape = (num_samples, num_output)
        self.X = X # Assume shape = (num_samples, num_inputs)
        self.YX = np.concatenate((Y, X), axis=-1) # Assume shape = (num_samples, num_output+num_inputs)
        self.num_Y = self.Y.shape[1]
        self.num_X = self.X.shape[1]

        # Fitting
        self.esti_mean_YX = np.mean(self.YX, axis = 0) # (num_output+num_inputs,)
        self.esti_covariance_matrix_YX = np.cov(self.YX.T, ddof=1) # (num_output+num_inputs, num_output+num_inputs)
        
        # Useful stats
        self.mu_Y = self.esti_mean_YX[:self.num_Y]
        self.mu_X = self.esti_mean_YX[self.num_Y:]
        self.cov_XX = self.esti_covariance_matrix_YX[self.num_Y:, self.num_Y:]
        self.cov_YY = self.esti_covariance_matrix_YX[:self.num_Y, :self.num_Y]
        self.cov_YX = self.esti_covariance_matrix_YX[:self.num_Y, self.num_Y:]
        self.cov_XY = self.esti_covariance_matrix_YX[self.num_Y:, :self.num_Y]
        
    def get_conditional_mean(self, alpha):
        # alpha = (num_samples, self.num_X)
        return (
            self.mu_Y + 
            np.linalg.multi_dot((
                self.cov_YX,
                np.linalg.inv(self.cov_XX),
                (alpha - self.mu_X).T))
        ).flatten()
        
    def get_conditional_cov(self):
        return (
            self.cov_YY - 
            np.linalg.multi_dot((
                self.cov_YX,
                np.linalg.inv(self.cov_XX), 
                self.cov_XY))
        ).flatten()[0]

class TruncatedConditionalGaussianDistribution:
    def __init__(self, Y, X, Y_Range):
        self.cgd = ConditionalGaussianDistribution(Y, X)
        self.Y_lb, self.Y_ub = Y_Range[0], Y_Range[1]

        # https://en.wikipedia.org/wiki/Truncated_normal_distribution
        self.cond_std = np.sqrt(self.cgd.get_conditional_cov())
        
    def get_conditional_mean(self, alpha):
        from scipy.stats import norm
        cond_mean = self.cgd.get_conditional_mean(alpha)
        alpha = (self.Y_lb-cond_mean)/self.cond_std
        beta = (self.Y_ub-cond_mean)/self.cond_std
        pdf_over_cdf = (
            (norm.pdf(beta)-norm.pdf(alpha))/
            (norm.cdf(beta)-norm.cdf(alpha))
        )
        mean_bias = -self.cond_std*pdf_over_cdf
        return cond_mean+mean_bias
        
    def get_conditional_cov(self, alpha):
        from scipy.stats import norm
        cond_mean = self.cgd.get_conditional_mean(alpha)
        cond_cov = self.cgd.get_conditional_cov()
        alpha = (self.Y_lb-cond_mean)/self.cond_std
        beta = (self.Y_ub-cond_mean)/self.cond_std
        pdf_over_cdf = (
            (norm.pdf(beta)-norm.pdf(alpha))/
            (norm.cdf(beta)-norm.cdf(alpha))
        )
        var_multiplier = (
            1-(
                (beta*norm.pdf(beta)-alpha*norm.pdf(alpha))/
                (norm.cdf(beta)-norm.cdf(alpha))
            )-np.square(pdf_over_cdf)
        )
        return cond_cov*var_multiplier
        
def get_entropy(pred_prob):
    from scipy.stats import entropy
    base = 2  # work in units of bits
    pred_prob = [1-pred_prob, pred_prob]
    return entropy(pred_prob, base=base, axis=0)

def get_incorrect_prob(true, pred_prob):
    pred_prob = np.array([1-pred_prob, pred_prob]).transpose()
    return pred_prob[range(len(true)), 1-true]

def get_uncertainty(pred_dict, target_col):
    fitted_df = pred_dict["val_df"]
    train_true = fitted_df['colorectal cancer'].values
    train_pred_prob = fitted_df['colorectal cancer_pred'].values
    train_entropy = get_entropy(train_pred_prob)
    train_incorrect_prob = get_incorrect_prob(train_true, train_pred_prob)

    # Fit Conditional Gaussian Distribution
    # uncertainty_distribution = ConditionalGaussianDistribution(
    #     Y=np.expand_dims(train_incorrect_prob, axis=1), 
    #     X=np.expand_dims(train_entropy, axis=1))
    uncertainty_distribution = TruncatedConditionalGaussianDistribution(
        Y=np.expand_dims(train_incorrect_prob, axis=1), 
        X=np.expand_dims(train_entropy, axis=1), Y_Range=[0,1])
    
    # Get uncertainty on all pred_dfs in pred_dict
    output_split_dict = {}
    for splitname, split_df in pred_dict.items():
        split_df = split_df.copy()
        true = split_df[target_col].values
        pred_prob = split_df[f'{target_col}_pred'].values
        pred_label = pred_prob.round()
        incorrect_prob = get_incorrect_prob(true, pred_prob)
        entropy = get_entropy(pred_prob)
        uncertainty_prob = uncertainty_distribution.get_conditional_mean(
            np.expand_dims(entropy, axis=1)
        )
        uncertainty_std = np.sqrt(uncertainty_distribution.get_conditional_cov(
            np.expand_dims(entropy, axis=1)
        ))
        uncertainty_prob_lb = uncertainty_prob - 1.96*uncertainty_std
        uncertainty_prob_ub = uncertainty_prob + 1.96*uncertainty_std
        split_df["entropy"] = entropy
        split_df["uncertainty_prob"] = uncertainty_prob
        split_df["uncertainty_prob_lb"] = uncertainty_prob_lb
        split_df["uncertainty_prob_ub"] = uncertainty_prob_ub
        split_df["incorrect_class_prob"] = incorrect_prob
        split_df[f"{target_col}_pred_label"] = pred_label
        output_split_dict[splitname] = split_df
        num_within_bounds = (
            (split_df["incorrect_class_prob"] <= split_df['uncertainty_prob_ub']) & 
            (split_df["incorrect_class_prob"] >= split_df['uncertainty_prob_lb'])).sum()
        print(f"Percentage of incorrect prob within bounds for {splitname}:", num_within_bounds/len(split_df))
    return output_split_dict

def calculate_aurc(ue, loss):
    n = len(ue)
    prop_list, loss_list = [], []
    percentiles = np.linspace(1, 100, num=100) 
    for percentile in percentiles:
        threshold = np.percentile(ue, percentile)
        confident_mask = ue<=threshold
        confident_proportion = np.sum(confident_mask)/n
        cur_loss = np.mean(loss[confident_mask])
        prop_list.append(confident_proportion)
        loss_list.append(cur_loss)
    return sum(loss_list)

def evaluate_cur_ue(pred_df, ue_col, loss_col, target_col):
    from scipy.stats import pearsonr
    ue_perf_list = []
    index_list = []
    for split_name, split_df in pred_df.groupby("split"):
        loss = split_df[loss_col]
        ue = split_df[ue_col]
        correct = split_df[target_col] == split_df[f"{target_col}_pred_label"]
        corr, _ = pearsonr(ue, loss)
        aurc = calculate_aurc(ue, loss)
        ue_perf_list.append({
            "Correlation": corr,
            "AURC": aurc,
            "Mean Loss":np.mean(loss),
            "Mean UE":np.mean(ue),
            "Mean UE Correct": np.mean(ue[correct]),
            "Mean UE Incorrect": np.mean(ue[np.logical_not(correct)])
        })
        index_list.append(split_name)
    print(ue_col)
    perf_df = pd.DataFrame(ue_perf_list)
    perf_df.index = index_list
    display(perf_df.loc[["Train", "Valid", "Test"]])

def get_ue_loss_scatterplot(pred_df, stat_dict, cutoff=None, nbins=10):
    eqn_label_fs = 7
    line_col = 'black'
    point_color = "#0090C1"
    point_size = 75
    point_alpha = 0.5
    marker = "."
    formatting_dict = dict(color=point_color, s=point_size, alpha=point_alpha, marker=marker, edgecolors='none')
    num_cols = len(stat_dict)
    bin_width = int(100/nbins)
    bin_edges = np.array([0]+list(np.linspace(bin_width, 100, num=10)/100))
    
    fig, axes = plt.subplots(2, num_cols, figsize=(num_cols*2, 1.5*2), dpi=300, sharey="row", sharex="col")
    for i, (ue_type, ue_info_dict) in enumerate(stat_dict.items()):
        ue_col, loss_col = ue_info_dict["ue"], ue_info_dict["loss"]
        ue_df = pred_df.copy()
        ue, loss = ue_df[ue_col], ue_df[loss_col]
        ue = (ue - ue.min()) / (ue.max() - ue.min())
        ue_df[ue_col] = ue
        
        # Plot scatter
        # axes[0, i].axvspan(0, bin_width/100, color='grey', alpha=0.2, linewidth=0)
        axes[0,i].scatter(ue, loss, **formatting_dict)
        # Plot line
        m, c = np.polyfit(ue, loss, 1) 
        ue = np.sort(ue)
        axes[0, i].plot(ue, m*ue+c, color=line_col, linestyle='-', label=f'y = {m:.3f}x + {c:.3f}', linewidth=1.5)
        axes[0, i].legend(fontsize=eqn_label_fs)
        if i == 0:
            axes[0, i].set_ylabel("Error")
            
        # Plot Barplot
        ue_df['bin'] = pd.cut(
            ue_df[ue_col], bins=bin_edges, labels=bin_edges[1:], include_lowest=True, right=True)
        grouped = ue_df.groupby("bin", observed=False)
        grouped_loss = grouped[loss_col].mean()
        # Comparison of losses
        axes[1,i].bar(
            bin_edges[1:]-bin_width/100/2, grouped_loss, width=bin_width/100*0.9, color="#0090C1")
        ticks = list(axes[1,i].get_yticks(minor=True)) + [grouped_loss[0.1]]
        axes[1,0].set_yticks(ticks, minor=True)
        if i == 0:
            axes[1,i].set_ylabel("Mean Error")
        axes[1,i].set_xlabel(ue_type)

    plt.tight_layout()

def get_risk_coverage_curve(pred_df, stat_dict, cutoff=None):
    fig, ax = plt.subplots(1, 1, figsize=(5, 1.5), dpi=300)
    if cutoff:
        plt.axvspan(0, cutoff, color='grey', alpha=0.2, linewidth=0)
    for ue_type, ue_info_dict in stat_dict.items():
        loss_list, thres_list, prop_list = [], [], []
        ue_col, loss_col = ue_info_dict["ue"], ue_info_dict["loss"]
        ue = pred_df[ue_col]
        loss = pred_df[loss_col]
        n = len(ue)
        thres_list = np.unique(ue)
        thres_list = np.sort(ue) # sorted in increasing order
        for threshold in thres_list:
            confident_mask = ue<=threshold
            confident_proportion = np.sum(confident_mask)/n
            cur_loss = np.mean(loss[confident_mask])
            prop_list.append(confident_proportion)
            loss_list.append(cur_loss)
        ax.plot(np.array(prop_list)*100, loss_list, label=ue_type, alpha=0.8)
    ax.set_xlabel("Coverage (%)")
    ax.set_ylabel("Prediction Error")
    if cutoff:
        ax.set_xticks(list(ax.get_xticks()) + [cutoff])
        ax.set_xlim(xmin=-5, xmax=105)
    ax.legend(
        loc='upper center', bbox_to_anchor=(0.5, -0.35), ncol=3)

def get_pred_df(pred_dict):
    df_list = []
    df_names = {"train_df":"Train", "val_df": "Valid", "test_df":"Test"}
    for df_label, df_name in df_names.items():
        cur_df = pred_dict[df_label]
        cur_df["split"] = df_name
        df_list.append(cur_df)
    return pd.concat(df_list)

def buffered_shap(explainer, matrix, chunksize, pred):
    def process_shap(shap, pred):
        pred = (pred > 0.50).values.astype(int)
        # (2069, 7, 2) -> (2069, 2, 7)
        shap = np.transpose(shap, (0, 2, 1))
        # Get Shap of Predicted
        shap = shap[range(len(shap)), pred]
        return shap
    
    from math import ceil
    all_exp = []
    matrix_len = len(matrix)
    num_chunks = ceil(matrix_len/chunksize)
    for chunk_i in tqdm(range(num_chunks), total=num_chunks):
        start = chunk_i*chunksize
        end = (chunk_i+1)*chunksize
        if end < matrix_len:
            matrix_chunk = matrix[chunk_i*chunksize:(chunk_i+1)*chunksize]
            pred_chunk = pred[chunk_i*chunksize:(chunk_i+1)*chunksize]
        else:
            matrix_chunk = matrix[chunk_i*chunksize:]
            pred_chunk = pred[chunk_i*chunksize:]
        shap_chunk = explainer.shap_values(matrix_chunk)
        exp_chunk = process_shap(shap_chunk, pred_chunk)
        all_exp.append(exp_chunk)
    return np.vstack(all_exp)

def get_shap_values(model, pred_dict, feat_cols, target_col):
    shap_cols = [f"{col}_shap" for col in feat_cols]
    explainer = shap.TreeExplainer(model)
    output_split_dict = {}
    for splitname, split_df in pred_dict.items():
        split_df = split_df.copy()
        pred = split_df[f"{target_col}_pred_label"]
        shap_values = buffered_shap(explainer, matrix=split_df[feat_cols].values, chunksize=100, pred=pred)
        split_df[shap_cols] = shap_values
        output_split_dict[splitname] = split_df
    return output_split_dict
        
def get_survey_information(model, pred_dict, feat_cols, target_col, stat_dict=None):
    # Get uncertainty estimate
    pred_dict = get_uncertainty(pred_dict, target_col)

    # Evaluate uncertainty estimates
    if stat_dict is not None:
        print("==== Uncertainty Performance ====")
        pred_df = get_pred_df(pred_dict)
        display(pred_df)
        # Show Performance Tables
        for key, values in stat_dict.items():
            evaluate_cur_ue(
                pred_df, ue_col=values["ue"], loss_col=values["loss"], target_col=target_col
            )
        # Show ue-loss scatterplot
        get_ue_loss_scatterplot(pred_dict["test_df"], stat_dict)
        # Show risk coverage curve
        get_risk_coverage_curve(pred_dict["test_df"], stat_dict)

    # Get shap values
    pred_dict = get_shap_values(model, pred_dict, feat_cols, target_col)

    return pred_dict

In [ ]:
stat_dict = {
    "Entropy":{"ue":"entropy", "loss":"incorrect_class_prob"},
    "Conditional Uncertainty":{"ue":"uncertainty_prob", "loss":"incorrect_class_prob"}
}
survey_pred_dict = get_survey_information(
    model=crc_clf_w_pc, pred_dict=crc_pred_dict_w_pc, 
    feat_cols=best_feat_w_pc_oversampled, 
    target_col=crc_target_col_new, 
    stat_dict=stat_dict
)

In [ ]:
survey_pred_dict["train_df"].iloc[:, -15:]

In [ ]:
joblib.dump(survey_pred_dict, "truncated_survey_pred_dict.joblib")

In [ ]:
survey_pred_dict = joblib.load("truncated_survey_pred_dict.joblib")
survey_pred_dict["val_df"]["uncertainty_prob"].describe()

In [ ]:
old_survey_pred_dict = joblib.load("survey_pred_dict.joblib")
old_survey_pred_dict["val_df"]["uncertainty_prob"].describe()

In [ ]:
def select_survey_information(
    survey_pred_dict, selection_list, feat_cols, target_col, scaler, continuous_feats, df_name="test_df"):
    shap_cols = [f"{col}_shap" for col in feat_cols]
    pred_label_col = f"{target_col}_pred_label"
    uncertainty_col = "uncertainty_prob"
    lb_col, ub_col = "uncertainty_prob_lb", "uncertainty_prob_ub"
    correct_col = "correct"
    important_cols = (
        list(feat_cols) + shap_cols + 
        [target_col, pred_label_col, correct_col, uncertainty_col, lb_col, ub_col]
    )
    df = survey_pred_dict[df_name]
    df = df.copy()   
    df[continuous_feats] = scaler.inverse_transform(df[continuous_feats].values)
    df[correct_col] = df[target_col] == df[pred_label_col]
    all_selected_rows = []
    for selection_dict in selection_list:
        # Select class
        cur_df = df[df[target_col] == selection_dict["target_col"]]
        # Select prediction outcome
        cur_df = cur_df.loc[df[correct_col] == selection_dict["correct"]]
        # Select certainty
        selection_fn = selection_dict["uncertainty_selection_fn"]
        selected_uncertainty = selection_fn(cur_df[uncertainty_col])
        selected_row = cur_df[cur_df[uncertainty_col]==selected_uncertainty].head(1)
        all_selected_rows.append(selected_row)
    output_df = pd.concat(all_selected_rows)
    replacement_dict = {1:"Cancer", 0:"Normal"}
    output_df[pred_label_col] = output_df[pred_label_col].astype(int)
    output_df[pred_label_col] = output_df[pred_label_col].replace(replacement_dict)
    output_df[target_col] = output_df[target_col].replace(replacement_dict)
    output_df.loc[output_df["uncertainty_prob_lb"]<0, "uncertainty_prob_lb"] = 0
    return output_df[important_cols]

selection_list = [
    {"target_col": 1, "correct": True, "uncertainty_selection_fn": min}, # Cancer, Correct, Certain
    {"target_col": 1, "correct": False, "uncertainty_selection_fn": max}, # Cancer, Incorrect, Uncertain
    {"target_col": 0, "correct": True, "uncertainty_selection_fn": min}, # Normal, Correct, Certain
    {"target_col": 0, "correct": False, "uncertainty_selection_fn": max}, # Normal, Incorrect, Uncertain
]
selected_df = select_survey_information(
    survey_pred_dict, selection_list, 
    feat_cols=best_feat_w_pc_oversampled, target_col=crc_target_col_new, 
    scaler=crc_scaler, continuous_feats = crc_continuous_cols, df_name="test_df"
)
selected_df

In [ ]:
def display_survey_questions(selected_df, feat_cols, clean_feats, clean_shap_feats, target_col):
    pred_col = f"{target_col}_pred_label"
    uncertainty_col = "uncertainty_prob"
    lb_col, ub_col = "uncertainty_prob_lb", "uncertainty_prob_ub"
    shap_cols = [col+"_shap" for col in feat_cols]
    def row_to_df(old_feat_cols, new_feat_cols, row):
        feat_values = row[old_feat_cols].values
        return pd.DataFrame({"Feature":new_feat_cols, "Value":feat_values})
        
    for i in range(len(selected_df)):
        # Show patient features
        row = selected_df.iloc[i]
        print("Index:", row.name)
        patient_feature_df = row_to_df(feat_cols, clean_feats, row)
        display(patient_feature_df)
        
        # Show prediction
        pred = row[pred_col]
        print("Prediction:", pred)

        # Show uncertainty estimate
        uncertainty_prob = row[uncertainty_col]*100
        lb, ub = row[lb_col]*100, row[ub_col]*100
        print(f"Uncertainty")
        print(f"Score: {uncertainty_prob:.1f}%")
        print(f"Bounds: {lb:.1f}-{ub:.1f}%")

        # Show SHAP
        colors = {
            "Protective Factors": "#6AD5CB",
            "Risk Factors":"#E3655B"
        }
        fig, ax = plt.subplots(1, 1, dpi=300, figsize=(4, 4))
        shap_df = row_to_df(shap_cols, clean_shap_feats, row)
        shap_df = shap_df.sort_values(by="Value")
        if pred=="Normal":
            color_vec = [
                colors["Risk Factors"] if shap < 0 else colors["Protective Factors"] 
                for shap in shap_df["Value"]]
        else:
            color_vec = [
                colors["Protective Factors"] if shap < 0 else colors["Risk Factors"] 
                for shap in shap_df["Value"]]
        ax.barh(shap_df["Feature"], shap_df["Value"], color=color_vec)
        ax.set_xlabel("Feature Attribution (SHAP)")
        labels, handles = [], []
        for label, color in colors.items():
            labels.append(label)
            handles.append(plt.Rectangle((0,0),1,1, color=color))
        plt.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)
        plt.tight_layout()
        plt.show()

        print()
        

clean_feats = [
    "Systolic blood pressure (SBP)", "Diastolic blood pressure (DBP)", 
    "DASH Diet Score (DASH)", "Body mass index (BMI)", "aMED Diet Score (aMED)", 
    "Leisure screen time (LST)", "Age"]
clean_shap_feats = ["SBP", "DBP", "DASH", "BMI", "aMED", "LST", "Age"]
display_survey_questions(
    selected_df, feat_cols=best_feat_w_pc_oversampled, target_col=crc_target_col_new, 
    clean_feats=clean_feats, clean_shap_feats=clean_shap_feats
)

In [ ]:
crc_pred_dict_w_pc = get_shap_values(**crc_pred_dict_w_pc, feat_cols=crc_feat_cols_w_pc, target_col=crc_target_col_new, 
    clf=crc_clf_w_pc)

In [ ]:
joblib.dump(crc_pred_dict_w_pc, "crc_pred_dict_w_pc.joblib")

In [ ]:
crc_pred_df_w_pc = format_results(
    **crc_pred_dict_w_pc, target_col=crc_target_col_new, scaler=crc_scaler, continuous_cols=crc_continuous_cols_new)
display(crc_pred_df_w_pc)

In [ ]:
def save_model_n_outputs(perf_df, pred_df, clf, name):
    fp_perf_pred_folder = os.path.join(fp_results_checkpoint_folder, name)
    if not os.path.exists(fp_perf_pred_folder):
        os.makedirs(fp_perf_pred_folder)
    fp_perf = os.path.join(fp_perf_pred_folder, "performance.csv")
    fp_pred = os.path.join(fp_perf_pred_folder, "prediction.csv")
    # Save perf and prediction
    perf_df.to_csv(fp_perf)
    pred_df.to_csv(fp_pred)
    # Save model
    fp_model = os.path.join(fp_model_checkpoint_folder, f"{name}.pickle")
    joblib.dump(clf, fp_model) 
    print("Saved!")
save_model_n_outputs(crc_perf_df_w_pc, crc_pred_df_w_pc, crc_clf_w_pc, name="crc_w_pc")

In [ ]:
def load_model_n_outputs(name):
    fp_perf_pred_folder = os.path.join(fp_results_checkpoint_folder, name)
    if not os.path.exists(fp_perf_pred_folder):
        print("Path does not exist!")
        return
    fp_perf = os.path.join(fp_perf_pred_folder, "performance.csv")
    fp_pred = os.path.join(fp_perf_pred_folder, "prediction.csv")
    perf_df = pd.read_csv(fp_perf, index_col=0)
    pred_df = pd.read_csv(fp_pred, index_col=0)
    fp_model = os.path.join(fp_model_checkpoint_folder, f"{name}.pickle")
    clf = joblib.load(fp_model)
    return perf_df, pred_df, clf
crc_perf_df_w_pc, crc_pred_df_w_pc, crc_clf_w_pc = load_model_n_outputs(name="crc_w_pc")
display(crc_perf_df_w_pc)
display(crc_pred_df_w_pc)

In [ ]:
crc_pred_df_w_pc["colorectal cancer_pred_bin"].value_counts()

In [ ]:
def visualise_global_shap_crc(pred_df, feat_cols, target_col):   
    columns_to_replace ={
        "Age_interview_shap": "Age_shap", 
        'aHEI2010score_shap':"aHEI2010 Dietary Score_shap", 
        "LeisureScreenTime_shap": "Leisure Screen Time_shap",
        "Sex (0=Male, 1=Female)_shap": "Female_shap",
        'alcohol_DailyandWeekly(1)vsMonthlyandNonDrinkers(0)_shap': "Frequent Drinker_shap",
        'smoke_ex(1)_shap': "Ex Smoker_shap",
       'smoke_current(2)_shap': "Current Smoker_shap"
    }
    class_label_list = ["Normal", "Cancer"]
    
    pred_df = pred_df.copy()
    shap_cols = [f"{col}_shap" for col in feat_cols]
    pred_bin_col = f"{target_col}_pred_bin"

    # Get correct predictions
    pred_df = pred_df[pred_df[target_col] == pred_df[pred_bin_col]]

    # Plot Shap Values
    fig, axes = plt.subplots(3, 2, figsize=(12, 10), dpi=300, sharey=True, sharex=True) #
    for i, split_name in enumerate(["train", "valid", "test"]):
        split_df = pred_df[pred_df["split"]==split_name]
        sorted_cols = split_df[shap_cols].abs().mean().sort_values().index
        for j, (class_label, class_df) in enumerate(split_df.groupby(target_col)):
            sorted_df = class_df[sorted_cols]
            sorted_df = sorted_df.rename(columns=columns_to_replace)
            axes[i, j].violinplot(sorted_df, showmedians=True, vert=False)
            axes[i, j].set_yticks(range(1, len(shap_cols)+1))
            axes[i, j].set_yticklabels([col[:-5] for col in sorted_df.columns])
            axes[i, j].set_title(f"{split_name}, {class_label_list[class_label]}")
            axes[i, j].set_xlabel("Shap")
            axes[i, j].axvline(0, color='grey')
    plt.tight_layout()
    
visualise_global_shap_crc(
    pred_df=crc_pred_df_w_pc, feat_cols=crc_feat_cols_w_pc, target_col=crc_target_col_new)

In [ ]:
def visualise_local_shap_crc(pred_df, feat_cols, target_col, condition_series, continuous_cols, categorical_cols, scaler):
    columns_to_replace = {
        "Age_interview": "Age", 
        'aHEI2010score':"aHEI2010 Dietary Score", 
        "LeisureScreenTime": "Leisure Screen Time",
        "Sex (0=Male, 1=Female)": "Female",
        'alcohol_DailyandWeekly(1)vsMonthlyandNonDrinkers(0)': "Frequent Drinker",
        'smoke_ex(1)': "Ex-Smoker",
       'smoke_current(2)': "Current Smoker"
    }
    shap_columns_to_replace = {
        "Age_interview_shap": "Age_shap", 
        'aHEI2010score_shap':"aHEI2010 Dietary Score_shap", 
        "LeisureScreenTime_shap": "Leisure Screen Time_shap",
        "Sex (0=Male, 1=Female)_shap": "Female_shap",
        'alcohol_DailyandWeekly(1)vsMonthlyandNonDrinkers(0)_shap': "Frequent Drinker_shap",
        'smoke_ex(1)_shap': "Ex-Smoker_shap",
       'smoke_current(2)_shap': "Current Smoker_shap"
    }
    cat_column_conversion = {
        "Female": ["Male", "Female"],
        "Frequent Drinker": ["Infrequent Drinker", "Frequent Drinker"],
        "Smoker": ["Never Smoked", "Ex-Smoker", "Current Smoker"]
    }
    new_feat_cols = [columns_to_replace[col] if col in columns_to_replace else col for col in feat_cols]
    new_cat_cols = [columns_to_replace[col] if col in columns_to_replace else col for col in categorical_cols]
    class_label_list = ["Normal", "Cancer"]
    
    pred_df = pred_df.copy()
    shap_cols = [f"{col}_shap" for col in feat_cols]
    pred_bin_col = f"{target_col}_pred_bin"

    # Change shap names
    pred_df = pred_df.rename(columns=shap_columns_to_replace)
    shap_cols = [col for col in pred_df.columns if "_shap" in col]
        
    # Get row
    row = pred_df[condition_series].iloc[0]

    # Plot Shap Values
    shaps = row[shap_cols].sort_values(ascending=True)
    fig, ax = plt.subplots(1, 1, dpi=300, figsize=(6, 4))
    display_shap_cols = [col[:-5] for col in shaps.index]
    ax.barh(display_shap_cols, shaps)
    ax.set_xlabel("SHAP")
    plt.tight_layout()

    feat_df = []
    # Rename value columns
    pred_df = pred_df.rename(columns=columns_to_replace)
    row = pred_df[condition_series].iloc[0]
    print("Index:", row.name, ", Split:", row["split"])
    feat_df.append({"Feature": "Class", "Value": class_label_list[row[target_col]]})
    smoker_flag = False # Mark if smoker status has been printed
    for col in display_shap_cols[::-1]:
        if col in new_cat_cols:
            if col in cat_column_conversion:
                feat_df.append({"Feature": cat_column_conversion[col][row[col]], "Value": ""})
            elif "Smoker" in col and not smoker_flag:
                if row["Ex-Smoker"] == 1:
                    feat_df.append({"Feature": cat_column_conversion['Smoker'][1], "Value": ""})
                elif row["Current Smoker"] == 1:
                    feat_df.append({"Feature": cat_column_conversion['Smoker'][2], "Value": ""})
                else:
                    feat_df.append({"Feature": cat_column_conversion['Smoker'][0], "Value": ""})
                smoker_flag = True
        else:
            feat_df.append({"Feature": col, "Value": round(row[col], 3)})
    feat_df = pd.DataFrame(feat_df)
    feat_df = feat_df.set_index('Feature')
    display(feat_df)
    feat_df.to_clipboard()
        
visualise_local_shap_crc(
    pred_df=crc_pred_df_w_pc, feat_cols=crc_feat_cols_w_pc, target_col=crc_target_col_new, 
    condition_series=(
        (crc_pred_df_w_pc["split"]=="test") & 
        (crc_pred_df_w_pc['Sex (0=Male, 1=Female)']==0) & 
        (crc_pred_df_w_pc['colorectal cancer_pred']>0.5) &
        (crc_pred_df_w_pc['colorectal cancer']>0.5)
    ),
    continuous_cols=crc_continuous_cols_new, categorical_cols=crc_categorical_cols_new, scaler=crc_scaler
)

In [ ]:
def get_balanced_statistics(pred_df, target_col, seed=seed):
    from sklearn.metrics import log_loss, accuracy_score, roc_auc_score
    pred_df = pred_df.copy()
    class_1_df = pred_df[(pred_df["split"]=="test") & (pred_df[target_col]==1)]
    class_0_df = pred_df[(pred_df["split"]=="test") & (pred_df[target_col]==0)]
    class_0_df = class_0_df.sample(len(class_1_df), random_state=seed)
    new_test_pred_df = pd.concat([class_0_df, class_1_df])
    pred_col = target_col + "_pred"
    
    perf_df = pd.DataFrame(dict(
        AUC = [
            roc_auc_score(new_test_pred_df[target_col], new_test_pred_df[pred_col])],
        Accuracy = [
            accuracy_score(new_test_pred_df[target_col], new_test_pred_df[pred_col]>0.5)],
        Loss = [
            log_loss(new_test_pred_df[target_col], new_test_pred_df[pred_col])], 
    ), index=["Test Undersampled"])

    return perf_df

def repeatedly_get_balanced_statistics(iterations, pred_df, target_col, seed=seed):
    all_perf_df = []
    for i in tqdm(range(iterations), total=iterations):
        all_perf_df.append(get_balanced_statistics(pred_df=pred_df, target_col=target_col, seed=seed+i))
    new_perf_df = pd.concat(all_perf_df).mean().to_frame().T
    new_perf_df.index = ["Test Undersampled"]
    return new_perf_df

repeatedly_get_balanced_statistics(
    iterations=1000, pred_df=crc_pred_df_w_pc, target_col=crc_target_col_new, seed=seed)


#### Without PC

In [ ]:
crc_best_estimators_wo_pc, crc_tuning_df_wo_pc = tune_model_crc(
    train_df=crc_split_dict_encoded["train_df"], 
    val_df=crc_split_dict_encoded["val_df"], 
    feat_cols=crc_feat_cols_wo_pc, 
    target_col=crc_target_col_new,
    n_estimators_list=[600, 650, 700, 750, 800], 
    seed=seed
)
print(f"Optimal Number of Estimators: {crc_best_estimators_wo_pc}")
display(crc_tuning_df_wo_pc)

In [ ]:
crc_tuning_df_wo_pc.to_csv(fp_crc_tuning_wo_pc_file)

In [ ]:
crc_best_estimators_wo_pc=700	

In [ ]:
crc_clf_wo_pc = train_model_crc(
    **crc_split_dict_encoded, feat_cols=crc_feat_cols_wo_pc, target_col=crc_target_col_new, 
    n_estimators=crc_best_estimators_wo_pc, seed=seed
)

In [ ]:
crc_pred_dict_wo_pc, crc_perf_df_wo_pc = get_predictions(
    **crc_split_dict_encoded, feat_cols=crc_feat_cols_wo_pc, target_col=crc_target_col_new, 
    clf=crc_clf_wo_pc)
display(crc_perf_df_wo_pc)

In [ ]:
crc_pred_dict_wo_pc = get_shap_values(
    **crc_pred_dict_wo_pc, feat_cols=crc_feat_cols_wo_pc, target_col=crc_target_col_new, 
    clf=crc_clf_wo_pc, chunksize=256)

In [ ]:
crc_pred_df_wo_pc = format_results(
    **crc_pred_dict_wo_pc, target_col=crc_target_col_new, scaler=crc_scaler, continuous_cols=crc_continuous_cols_new)
display(crc_pred_df_wo_pc)

In [ ]:
save_model_n_outputs(crc_perf_df_wo_pc, crc_pred_df_wo_pc, crc_clf_wo_pc, name="crc_wo_pc")

In [ ]:
crc_perf_df_wo_pc, crc_pred_df_wo_pc, crc_clf_wo_pc = load_model_n_outputs(name="crc_wo_pc")
display(crc_perf_df_wo_pc)
display(crc_pred_df_wo_pc)

In [ ]:
crc_pred_df_w_pc["colorectal cancer_pred_bin"].value_counts()

In [ ]:
visualise_global_shap_crc(
    pred_df=crc_pred_df_wo_pc, feat_cols=crc_feat_cols_wo_pc, target_col=crc_target_col_new)

In [ ]:
visualise_local_shap_crc(
    pred_df=crc_pred_df_wo_pc, feat_cols=crc_feat_cols_wo_pc, target_col=crc_target_col_new, 
    condition_series=(
        (crc_pred_df_wo_pc["split"]=="test") & 
        (crc_pred_df_wo_pc['Sex (0=Male, 1=Female)']==0) & 
        (crc_pred_df_wo_pc['colorectal cancer_pred']>0.5) &
        (crc_pred_df_wo_pc['colorectal cancer']>0.5)
    ),
    continuous_cols=crc_continuous_cols_new, categorical_cols=crc_categorical_cols_new, scaler=crc_scaler
)

In [ ]:
repeatedly_get_balanced_statistics(
    iterations=1000, pred_df=crc_pred_df_wo_pc, target_col=crc_target_col_new, seed=seed)